# שבוע 08: ניתוח מתאר אליפטי (EFA) — מבוא

**Elliptic Fourier Analysis** מנתחת **מתאר סגור** ולא נקודות ציון.  
היא מפרקת כל מתאר לסדרה של הרמוניות — כמו פורייה לצורות.

**מטרות השיעור:**
- הבנת EFA ועקרון ההרמוניות
- שחזור מתאר ממספר הרמוניות גדל
- טעינת נתוני קרדומות מ-MorphoJ
- PCA ראשוני של EFA

In [ ]:
!pip install pyefd python-bidi -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
import pyefd
rtl = get_display
print('הכל מוכן!')

## הדגמה: מתאר סינתטי

ניצור מתאר של כוכב ונראה כיצד EFA מחשב את המקדמים.

In [ ]:
# מתאר כוכב
n_pts = 200
theta = np.linspace(0, 2*np.pi, n_pts, endpoint=False)
r = 1.0 + 0.4 * np.cos(5 * theta)  # 5 עלי כותרת
contour_star = np.column_stack([r * np.cos(theta), r * np.sin(theta)]) * 100

# EFA
coeffs_full = pyefd.elliptic_fourier_descriptors(contour_star, order=20, normalize=True)
print(f'צורת מטריצת המקדמים: {coeffs_full.shape}  (harmonics × 4 coefficients)')
print('4 מקדמים לכל הרמוניקה: a_n, b_n, c_n, d_n')
print()
print('המקדמים הראשונים:')
for i, row in enumerate(coeffs_full[:5]):
    print(f'  הרמוניקה {i+1}: a={row[0]:.3f}, b={row[1]:.3f}, c={row[2]:.3f}, d={row[3]:.3f}')

## שחזור מתאר: השפעת מספר ההרמוניות

נשחזר את הכוכב עם 1, 2, 4, 8 ו-20 הרמוניות.  
שימו לב ל-API הנכון: `pyefd.reconstruct_contour(coeffs_mat, locus=(0,0), num_points=400)`

In [ ]:
orders = [1, 2, 4, 8, 20]
fig, axes = plt.subplots(1, 5, figsize=(16, 4))

for ax, order in zip(axes, orders):
    # לקחת את coeffs_full[:order] — מטריצה בצורת (order, 4)
    coeffs_mat = coeffs_full[:order]  # (order, 4)
    reconstructed = pyefd.reconstruct_contour(coeffs_mat, locus=(0, 0), num_points=400)
    ax.plot(reconstructed[:, 0], reconstructed[:, 1], 'steelblue', lw=1.5)
    ax.plot(contour_star[:, 0], contour_star[:, 1], 'gray', lw=0.8, alpha=0.4, linestyle='--')
    ax.set_title(f'{order} הרמוניות', fontsize=10)
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle(rtl('שחזור מתאר כוכב: מספר הרמוניות גדל'), fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def parse_efa_dat(text_or_path):
    """Parse MorphoJ EFA dat: tab-sep, European decimals, 2 header lines."""
    if '\n' in text_or_path:
        lines = text_or_path.strip().split('\n')
    else:
        with open(text_or_path, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    groups, data, sizes = [], [], []
    for line in lines[2:]:
        line = line.strip() if hasattr(line, 'strip') else line
        if not line:
            continue
        cols = line.split('\t')
        groups.append(cols[3].strip())
        sizes.append(float(cols[4].replace(',', '.').replace('E', 'e')))
        coeffs = []
        for c in cols[5:]:
            c = c.strip()
            if c and c != '-':
                try:
                    coeffs.append(float(c.replace(',', '.').replace('E', 'e')))
                except:
                    pass
        data.append(coeffs)
    min_len = min(len(d) for d in data)
    return np.array([d[:min_len] for d in data]), np.array(groups), np.array(sizes)

print('parse_efa_dat מוכן')

## טעינת נתוני קרדומות

קובץ `axes_efa.dat` מכיל 60 קרדומות (30 G3, 30 G4) עם 30 הרמוניות = 120 מקדמי EFA.

In [ ]:
import urllib.request

AXES_URL = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/axes/axes_efa.dat'

try:
    with urllib.request.urlopen(AXES_URL, timeout=15) as r:
        text = r.read().decode('utf-8', errors='replace')
    efa_data, groups, sizes = parse_efa_dat(text)
    print(f'נטענו {len(efa_data)} קרדומות, {efa_data.shape[1]} מקדמי EFA')
    print(f'קבוצות: {np.unique(groups, return_counts=True)}')
    DATA_OK = True
except Exception as e:
    print(f'שגיאה: {e} — משתמשים בנתונים סינתטיים')
    np.random.seed(42)
    n_g3, n_g4 = 30, 30
    # G3: יותר עגול (harmonic 1 גדול), G4: יותר מוארך
    base = np.random.randn(n_g3 + n_g4, 120) * 0.1
    base[:n_g3, 0] += 0.5
    base[n_g3:, 2] += 0.4
    efa_data = base
    groups = np.array(['G3']*n_g3 + ['G4']*n_g4)
    sizes = np.random.uniform(50, 150, n_g3+n_g4)
    sizes[:n_g3] += 20
    DATA_OK = False

print(f'מקור נתונים: {"GitHub" if DATA_OK else "Synthetic"}')

## PCA על נתוני EFA של הקרדומות

In [ ]:
from sklearn.decomposition import PCA

pca_axes = PCA()
scores_axes = pca_axes.fit_transform(efa_data)
ve_axes = pca_axes.explained_variance_ratio_ * 100

print(f'PC1: {ve_axes[0]:.1f}%, PC2: {ve_axes[1]:.1f}%')

fig, ax = plt.subplots(figsize=(8, 6))
mask_g3 = groups == 'G3'
mask_g4 = groups == 'G4'

ax.scatter(scores_axes[mask_g3, 0], scores_axes[mask_g3, 1],
           c='steelblue', s=80, alpha=0.85, label=f'G3 (n={mask_g3.sum()})', zorder=3)
ax.scatter(scores_axes[mask_g4, 0], scores_axes[mask_g4, 1],
           c='darkorange', s=80, marker='s', alpha=0.85,
           label=f'G4 (n={mask_g4.sum()})', zorder=3)

ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel(f'PC1 ({ve_axes[0]:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({ve_axes[1]:.1f}%)', fontsize=12)
ax.set_title(rtl('מרחב הצורה: קרדומות מסוג G3 vs G4'), fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## סיכום

- **EFA** מפרקת מתאר סגור ל-4 מקדמים לכל הרמוניקה (a, b, c, d)
- **שחזור**: `pyefd.reconstruct_contour(coeffs_mat, locus=(0,0), num_points=400)`
- **נתוני קרדומות**: 60 דגימות, 120 מקדמים (30 הרמוניות × 4)
- **PCA**: מאפשר ראייה כוללת של מרחב הצורה

**שאלות לחשיבה:**
1. כמה הרמוניות צריך כדי לשחזר מתאר מורכב?
2. האם G3 ו-G4 נפרדים בבירור ב-PCA? מה זה אומר על הצורה?
3. מה ההבדל בין EFA לניתוח נקודות ציון?